In [ ]:
#better running code with ardino 
import cv2
import time
import winsound
import serial.tools.list_ports

# Load cascades
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades +
                                     'haarcascade_frontalface_default.xml')
eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades +
                                    'haarcascade_eye.xml')

# Camera
cap = cv2.VideoCapture(0)
cap.set(3, 640)
cap.set(4, 480)

# Arduino (optional)
ser = None
def init_arduino():
    global ser
    ports = serial.tools.list_ports.comports()
    for port in ports:
        try:
            ser = serial.Serial(port.device, 9600, timeout=1)
            time.sleep(2)
            print(f"Arduino connected: {port.device}")
            return
        except:
            pass
    print("Running without Arduino")

init_arduino()

def send_to_arduino(cmd):
    if ser:
        try:
            ser.write(cmd.encode())
        except:
            pass

# Parameters (tuned)
EYE_CLOSED_TIME = 1
ALERT_COOLDOWN = 1.5
EMERGENCY_TIME = 2

# State
eye_closed_start = None
last_alert_time = 0
emergency_triggered = False

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Preprocessing
    gray = cv2.equalizeHist(gray)

    faces = face_cascade.detectMultiScale(gray, 1.2, 5)

    status = "AWAKE"
    color = (0, 255, 0)

    eyes_detected = False  # RESET every frame

    for (x, y, w, h) in faces:
        cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)

        roi_gray = gray[y:y+h, x:x+w]
        roi_color = frame[y:y+h, x:x+w]

        eyes = eye_cascade.detectMultiScale(roi_gray, 1.1, 6)

        if len(eyes) >= 1:
            eyes_detected = True

        for (ex, ey, ew, eh) in eyes:
            cv2.rectangle(roi_color, (ex, ey), (ex+ew, ey+eh), (0, 255, 0), 2)

    # 🧠 TIME-BASED LOGIC (FIXED)
    current_time = time.time()

    if not eyes_detected:
        if eye_closed_start is None:
            eye_closed_start = current_time
    else:
        eye_closed_start = None
        emergency_triggered = False

    elapsed = 0
    if eye_closed_start:
        elapsed = current_time - eye_closed_start

    # ⚠️ WARNING
    if elapsed > EYE_CLOSED_TIME:
        status = "WARNING"
        color = (0, 255, 255)

        if current_time - last_alert_time > ALERT_COOLDOWN:
            winsound.Beep(1200, 300)
            send_to_arduino('W')
            last_alert_time = current_time

    # 🚨 EMERGENCY
    if elapsed > EMERGENCY_TIME:
        status = "EMERGENCY"
        color = (0, 0, 255)

        if not emergency_triggered:
            winsound.Beep(800, 800)
            send_to_arduino('S')
            emergency_triggered = True

    # UI
    cv2.putText(frame, f"STATUS: {status}", (30, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, color, 3)

    cv2.putText(frame, f"Closed Time: {elapsed:.1f}s", (30, 90),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)

    cv2.imshow("Driver Drowsiness System (Final)", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

if ser:
    ser.close()

In [ ]:
#final software code
import cv2
import time
import winsound

# Load cascades
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades +
                                     'haarcascade_frontalface_default.xml')
eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades +
                                    'haarcascade_eye.xml')

# Camera setup
cap = cv2.VideoCapture(0)
cap.set(3, 640)
cap.set(4, 480)

# Parameters
EYE_CLOSED_TIME = 0.8   # seconds → warning
EMERGENCY_TIME = 2   # seconds → strong alert
ALERT_COOLDOWN = 1.5      # avoid beep spam

# State variables
eye_closed_start = None
last_alert_time = 0
emergency_triggered = False

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Improve visibility
    gray = cv2.equalizeHist(gray)

    faces = face_cascade.detectMultiScale(gray, 1.2, 5)

    status = "AWAKE"
    color = (0, 255, 0)

    eyes_detected = False

    for (x, y, w, h) in faces:
        cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)

        roi_gray = gray[y:y+h, x:x+w]
        roi_color = frame[y:y+h, x:x+w]

        eyes = eye_cascade.detectMultiScale(roi_gray, 1.1, 6)

        if len(eyes) >= 1:
            eyes_detected = True

        for (ex, ey, ew, eh) in eyes:
            cv2.rectangle(roi_color, (ex, ey), (ex+ew, ey+eh), (0, 255, 0), 2)

    current_time = time.time()

    # Eye closure tracking
    if not eyes_detected:
        if eye_closed_start is None:
            eye_closed_start = current_time
    else:
        eye_closed_start = None
        emergency_triggered = False

    elapsed = 0
    if eye_closed_start:
        elapsed = current_time - eye_closed_start

    # ⚠️ Warning stage
    if elapsed > EYE_CLOSED_TIME:
        status = "WARNING"
        color = (0, 255, 255)

        if current_time - last_alert_time > ALERT_COOLDOWN:
            winsound.Beep(1200, 300)
            last_alert_time = current_time

    # 🚨 Emergency stage
    if elapsed > EMERGENCY_TIME:
        status = "DROWSY!"
        color = (0, 0, 255)

        if not emergency_triggered:
            winsound.Beep(800, 800)
            emergency_triggered = True

    # UI display
    cv2.putText(frame, f"STATUS: {status}", (30, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, color, 3)

    cv2.putText(frame, f"Eye Closed: {elapsed:.1f}s", (30, 90),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)

    cv2.imshow("Driver Drowsiness Detection (Software Only)", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
#v2
# 🚛 TRUCK DRIVER SLEEP DETECTION - COMPLETE SYSTEM
# Single cell for Jupyter Notebook - FULLY FUNCTIONAL!

import cv2
import dlib
import numpy as np
from imutils import face_utils
import time
from IPython.display import display, Javascript, HTML
from google.colab.output import eval_js  # For webcam
import base64
from datetime import datetime

# Install if needed (run once)
!pip install opencv-python dlib imutils -q

print("🔄 Loading face detection model...")
detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor("shape_predictor_68_face_landmarks.dat")

# Download model if missing
import os
if not os.path.exists("shape_predictor_68_face_landmarks.dat"):
    !wget http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
    !bunzip2 shape_predictor_68_face_landmarks.dat.bz2

# CONFIG - TRUCK SAFETY
EYE_AR_THRESH = 0.27        # Sleep threshold
EYE_AR_CONSEC_FRAMES = 24   # Confirmation frames
COUNTER = 0
TOTAL_ALERTS = 0
ALERT_LEVEL = 0

def eye_aspect_ratio(eye):
    """Calculate Eye Aspect Ratio (EAR)"""
    A = np.linalg.norm(eye[1] - eye[5])  # Top eyelid
    B = np.linalg.norm(eye[2] - eye[4])  # Middle
    C = np.linalg.norm(eye[0] - eye[3])  # Eye width
    ear = (A + B) / (2.0 * C)
    return ear

def create_alert_sound():
    """Beep alert for Jupyter"""
    print("\a")  # Terminal bell
    print("🚨" * 10)

# MAIN DETECTION FUNCTION
def detect_drowsiness(frame):
    global COUNTER, TOTAL_ALERTS, ALERT_LEVEL
    
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    rects = detector(gray, 0)
    
    # No face = warning
    if len(rects) == 0:
        cv2.putText(frame, "😶 FACE NOT DETECTED", (50, 50), 
                   cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
        return frame
    
    for rect in rects:
        # Get facial landmarks
        shape = predictor(gray, rect)
        shape = face_utils.shape_to_np(shape)
        
        # Extract eyes
        leftEye = shape[face_utils.FACIAL_LANDMARKS_IDXS["left_eye"][0]:face_utils.FACIAL_LANDMARKS_IDXS["left_eye"][1]]
        rightEye = shape[face_utils.FACIAL_LANDMARKS_IDXS["right_eye"][0]:face_utils.FACIAL_LANDMARKS_IDXS["right_eye"][1]]
        
        # Calculate EAR
        leftEAR = eye_aspect_ratio(leftEye)
        rightEAR = eye_aspect_ratio(rightEye)
        ear = (leftEAR + rightEAR) / 2.0
        
        # Draw eyes
        cv2.drawContours(frame, [leftEye], -1, (0, 255, 0), 2)
        cv2.drawContours(frame, [rightEye], -1, (0, 255, 0), 2)
        
        # SLEEP DETECTION LOGIC
        if ear < EYE_AR_THRESH:
            COUNTER += 1
            
            # VISUAL WARNING
            cv2.putText(frame, "⚠️ EYES CLOSING", (50, 80), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
            
            # ALERT LEVELS
            if COUNTER >= EYE_AR_CONSEC_FRAMES:
                ALERT_LEVEL = min(3, COUNTER // 30)
                TOTAL_ALERTS += 1
                
                # EMERGENCY ALERT
                if ALERT_LEVEL >= 2:
                    color = (0, 0, 255)
                    text = f"🚨 SLEEP DETECTED - LEVEL {ALERT_LEVEL}"
                    create_alert_sound()
                    print(f"🚨 ALERT #{TOTAL_ALERTS} - {datetime.now().strftime('%H:%M:%S')}")
                else:
                    color = (0, 255, 255)
                    text = "⚠️ WARNING - Wake up!"
                
                cv2.putText(frame, text, (50, 120), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)
        else:
            COUNTER = 0
            ALERT_LEVEL = 0
        
        # DASHBOARD
        cv2.rectangle(frame, (10, 200), (400, 320), (50, 50, 50), 2)
        cv2.putText(frame, f"EAR: {ear:.3f}", (20, 230), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
        cv2.putText(frame, f"COUNTER: {COUNTER}/{EYE_AR_CONSEC_FRAMES}", (20, 260), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
        cv2.putText(frame, f"ALERTS: {TOTAL_ALERTS}", (20, 290), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    
    return frame

# WEBCAM ACCESS IN JUPYTER
from IPython.display import display, Javascript
import base64

def take_photo(filename='photo.jpg', quality=0.8):
    js = Javascript('''
        async function takePhoto(quality) {
            const div = document.createElement('div');
            const capture = document.createElement("button");
            capture.textContent = "Capture";
            div.appendChild(capture);

            const video = document.createElement('video');
            video.style.display = 'block';
            const stream = await navigator.mediaDevices.getUserMedia({video: true});

            document.body.appendChild(div);
            div.appendChild(video);
            video.srcObject = stream;
            await video.play();

            // Resize the output to fit the video element.
            google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

            // Wait for Capture to be clicked.
            await new Promise((resolve) => capture.onclick = resolve);

            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);
            stream.getVideoTracks()[0].stop();
            div.remove();
            return canvas.toDataURL('image/jpeg', quality);
        }
    ''')
    display(js)
    data = eval_js('takePhoto({})'.format(quality))
    
    # decode base64 image
    binary = base64.b64decode(data.split(',')[1])
    with open(filename, "wb") as f:
        f.write(binary)
    return filename

# 🔥 LIVE WEBCAM DETECTION LOOP
print("🎥 Starting LIVE detection...")
print("👀 Look at camera and CLOSE EYES to test!")
print("📊 Watch EAR value drop below 0.27")

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

while True:
    ret, frame = cap.read()
    if not ret:
        print("❌ Camera disconnected!")
        break
    
    # FLIP for mirror effect
    frame = cv2.flip(frame, 1)
    
    # DETECT SLEEP
    frame = detect_drowsiness(frame)
    
    # SHOW RESULT
    cv2.imshow('🚛 TRUCK SLEEP DETECTOR - Press Q to quit', frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

print(f"\n✅ SESSION COMPLETE!")
print(f"📈 Total Alerts Triggered: {TOTAL_ALERTS}")
print(f"😴 Sleep Threshold: {EYE_AR_THRESH}")
print("🎉 Ready for hardware deployment!")